# 00 — Exploration des données RSNA Pneumonia

Première partie du projet : **récupérer, comprendre et préparer** le dataset [RSNA Pneumonia Detection Challenge](https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge).

Ce notebook :
1. décompresse l'archive Kaggle téléchargée dans `data/rsna/`,
2. charge les labels et la distribution des classes,
3. **mappe les 3 classes RSNA vers les 3 sorties du contrat** (`normal`, `suspected_opacity`, `uncertain`),
4. lit les images DICOM,
5. visualise des radios avec leurs *bounding boxes* (bonus localisation),
6. exporte un `rsna_index.csv` propre, aligné sur le schéma `synthetic_cases.csv`.

> ⚠️ Données publiques dé-identifiées. Ne jamais committer les images (`.gitignore` est déjà configuré). Prototype éducatif, **pas** un dispositif médical.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pydicom

# Racine du projet (le notebook est dans notebooks/)
ROOT = Path('..').resolve()
RSNA = ROOT / 'data' / 'rsna'
print('Dossier RSNA :', RSNA)
print('Contenu      :', sorted(p.name for p in RSNA.iterdir()))

## 1. Décompression de l'archive

Le téléchargement Kaggle produit un gros `.zip` (et parfois des `.csv.zip`). On décompresse tout dans `data/rsna/`. Idempotent : on saute ce qui est déjà extrait.

In [ ]:
def unzip_all(folder: Path):
    for z in sorted(folder.glob('*.zip')):
        with zipfile.ZipFile(z) as zf:
            members = zf.namelist()
            # Évite de re-extraire si la cible existe déjà
            already = all((folder / m).exists() for m in members[:5])
            if already and len(members) > 5:
                print(f'  (déjà extrait) {z.name}')
                continue
            print(f'  extraction de {z.name} ({len(members)} fichiers)...')
            zf.extractall(folder)
    print('Décompression terminée.')

unzip_all(RSNA)
print(sorted(p.name for p in RSNA.iterdir() if not p.name.endswith('.zip')))

## 2. Chargement des labels

- `stage_2_train_labels.csv` : une ligne par *bounding box*. Colonnes `patientId, x, y, width, height, Target` (Target=1 si opacité). Un patient peut avoir plusieurs lignes ; les cas négatifs ont des coordonnées vides.
- `stage_2_detailed_class_info.csv` : la classe détaillée par patient (`Normal`, `Lung Opacity`, `No Lung Opacity / Not Normal`).

In [ ]:
labels = pd.read_csv(RSNA / 'stage_2_train_labels.csv')
classes = pd.read_csv(RSNA / 'stage_2_detailed_class_info.csv')

print('labels :', labels.shape)
display(labels.head())
print('classes :', classes.shape)
display(classes.head())

In [ ]:
# Distribution des classes détaillées (par ligne, doublons de patients inclus)
dist = classes['class'].value_counts()
print(dist)

ax = dist.plot(kind='bar', figsize=(7,4), color=['#4c72b0','#dd8452','#55a868'])
ax.set_title('Distribution des classes RSNA')
ax.set_ylabel('nombre de lignes')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 3. Mapping vers le contrat du projet

| Classe RSNA | Sortie projet |
|---|---|
| `Normal` | `normal` |
| `Lung Opacity` | `suspected_opacity` |
| `No Lung Opacity / Not Normal` | `uncertain` |

On construit une table propre, **un patient par ligne**, avec la sortie projet et le nombre de boîtes.

In [ ]:
RSNA_TO_PROJECT = {
    'Normal': 'normal',
    'Lung Opacity': 'suspected_opacity',
    'No Lung Opacity / Not Normal': 'uncertain',
}

# 1 ligne par patient pour la classe
per_patient = classes.drop_duplicates('patientId').copy()
per_patient['label'] = per_patient['class'].map(RSNA_TO_PROJECT)

# nombre de boîtes par patient
n_boxes = labels.groupby('patientId')['Target'].sum().rename('n_boxes')
per_patient = per_patient.merge(n_boxes, on='patientId', how='left')

print(per_patient['label'].value_counts())
display(per_patient.head())

## 4. Lecture d'une image DICOM

Les images sont en `.dcm`. On utilise `pydicom` pour extraire le tableau de pixels (niveaux de gris).

In [ ]:
TRAIN_IMG = RSNA / 'stage_2_train_images'

def load_dicom(patient_id: str) -> np.ndarray:
    ds = pydicom.dcmread(TRAIN_IMG / f'{patient_id}.dcm')
    return ds.pixel_array

pid = per_patient['patientId'].iloc[0]
img = load_dicom(pid)
print('patient :', pid, '| shape :', img.shape, '| dtype :', img.dtype,
      '| min/max :', img.min(), img.max())

plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')
plt.title(f'{pid}')
plt.axis('off')
plt.show()

## 5. Visualisation avec bounding boxes (bonus localisation)

On affiche quelques cas `Lung Opacity` (= `suspected_opacity`) avec leurs boîtes rouges. C'est le matériau pour la piste **COULD** de localisation visuelle.

In [ ]:
import matplotlib.patches as patches

def show_with_boxes(patient_id: str, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5,5))
    ax.imshow(load_dicom(patient_id), cmap='gray')
    boxes = labels[(labels.patientId == patient_id) & (labels.Target == 1)]
    for _, b in boxes.iterrows():
        ax.add_patch(patches.Rectangle((b.x, b.y), b.width, b.height,
                                       linewidth=2, edgecolor='red', facecolor='none'))
    cls = per_patient.loc[per_patient.patientId == patient_id, 'class'].iloc[0]
    ax.set_title(f'{patient_id[:8]}… | {cls}', fontsize=9)
    ax.axis('off')

opacity_ids = per_patient.loc[per_patient['class'] == 'Lung Opacity', 'patientId'].head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for pid, ax in zip(opacity_ids, axes.ravel()):
    show_with_boxes(pid, ax)
plt.tight_layout()
plt.show()

## 6. Export d'un index propre

On sauvegarde `data/rsna_index.csv`, aligné sur le schéma de `synthetic_cases.csv` (`case_id, image_path, source, label, split, quality, notes`) pour pouvoir réutiliser le reste du pipeline du projet.

In [ ]:
rng = np.random.default_rng(42)
split = rng.choice(['train', 'val', 'test'], size=len(per_patient), p=[0.7, 0.15, 0.15])

index = pd.DataFrame({
    'case_id': per_patient['patientId'].values,
    'image_path': [f'data/rsna/stage_2_train_images/{p}.dcm' for p in per_patient['patientId']],
    'source': 'RSNA Pneumonia Detection Challenge (Kaggle, 2018)',
    'label': per_patient['label'].values,
    'split': split,
    'quality': 'ok',
    'notes': 'n_boxes=' + per_patient['n_boxes'].fillna(0).astype(int).astype(str),
})

out = ROOT / 'data' / 'rsna_index.csv'
index.to_csv(out, index=False)
print('Écrit :', out, '|', len(index), 'cas')
display(index.head())
print(index['label'].value_counts())

## Prochaines étapes

- **Baseline** : classifieur opacité oui/non sur les images converties (DICOM → PNG normalisé).
- **Garde-fous** : seuil de confiance → renvoyer `uncertain` quand le modèle hésite.
- **Évaluation** : matrice de confusion sur les 3 classes, analyse d'erreurs.
- **COULD** : détection des boîtes (mAP/IoU), Grad-CAM comparé aux vraies boîtes.